[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabrielbelo2007/RP_PIGuard/blob/main/Notebooks/second_train/eval_table1.ipynb)

# PIGuard - Validação Comparativa Geral (Tabela 1 do Artigo) + Variantes train_augmented_mof

Este notebook executa a validação e reprodução experimental completa da **Tabela 1 do artigo PIGuard / InjecGuard (ACL 2025)**:

### 1. Modelos Avaliados:
- **Prompt Guard Models (Open-Weights)**:
  1. `Fmops (fmops, 2024)`: `fmops/distilbert-prompt-injection`
  2. `Deepset (Deepset, 2024b)`: `deepset/deberta-v3-base-injection`
  3. `PromptGuard (Meta, 2024)`: `meta-llama/Prompt-Guard-86M` (com fallback automático para mirror aberto)
  4. `ProtectAIv2 (ProtectAI.com, 2024)`: `ProtectAI/deberta-v3-base-prompt-injection-v2`
  5. `LakeraGuard (LakeraAI, 2024a)`: Referência oficial do artigo (API proprietária ignorada na execução conforme solicitado)

- **Large Language Models (via OpenRouter API / GPU)**:
  6. `GPT-4o (OpenAI, 2024a)`: Avaliado com o prompt oficial *Eliezer Yudkowsky* via OpenRouter API (com cache incremental)
  7. `Llama-2-chat (Touvron et al., 2023)`: Avaliado via OpenRouter API
  8. `LlamaGuard3 (Dubey et al., 2024)`: Avaliado via OpenRouter API

- **PIGuard (Ours - Oficial HF + Variantes train_augmented_mof)**:
  9. `PIGuard (Artigo - Referência Oficial)`: Valores oficiais reportados no paper
  10. `PIGuard (Hugging Face - leolee99/PIGuard)`: Inferência ao vivo do modelo oficial dos autores
  11. `PIGuard (train_augmented_mof_original)`: Vertente 1 (Original do Artigo)
  12. `PIGuard (train_augmented_mof_semi_rp)`: Vertente 2 (Semi-Reprodução)
  13. `PIGuard (train_augmented_mof_rp)`: Vertente 3 (Reproduzida)

### 2. Métricas Analisadas:
- **Performance**: Over-defense (%) [NotInject], Benign (%) [WildGuard], Malicious (%) [BIPIA Text/Code], Average (%)
- **Overhead**: GFLOPs, Inference Latency (ms), Efficiency (Average / Inference ms)


In [ ]:
# Célula 1: Verificação de Hardware
!nvidia-smi

import torch
print("\n--- Informações de Hardware e PyTorch ---")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# Célula 2: Montagem do Google Drive, Repositório e Dependências
import os
import sys
import shutil

# 1. Montagem do Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/RP_PIGuard_outputs"
print(f"Diretório de checkpoints no Google Drive: {DRIVE_OUTPUT_DIR}")

# 2. Localização ou sincronização do repositório
REPO_DIR = "/content/RP_PIGuard"
if not os.path.exists(REPO_DIR):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass

    if not token and os.path.exists("/content/drive/MyDrive/RP_PIGuard"):
        print("Copiando repositório existente do Google Drive...")
        shutil.copytree("/content/drive/MyDrive/RP_PIGuard", REPO_DIR)
    else:
        repo_url = f"https://{token}@github.com/gabrielbelo2007/RP_PIGuard.git" if token else "https://github.com/gabrielbelo2007/RP_PIGuard.git"
        print(f"Clonando repositório em {REPO_DIR}...")
        !git clone {repo_url} {REPO_DIR}
else:
    print(f"Repositório já presente em {REPO_DIR}. Atualizando...")
    !cd {REPO_DIR} && git pull origin main || true

%cd {REPO_DIR}

# 3. Instalar dependências necessárias
!pip install -q "transformers>=4.44.0,<5.0.0" "datasets>=2.21.0" sentencepiece protobuf tiktoken pyyaml pandas matplotlib seaborn tqdm

for p in [REPO_DIR, "/content/drive/MyDrive/RP_PIGuard", os.getcwd()]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

# 4. Recuperação Automática de Resultados Prévios do Google Drive (Economia de Custos)
os.makedirs("logs/Table1", exist_ok=True)
SYNC_FILES = [
    "table1_validation_raw.json",
    "table1_validation_summary.csv",
    "table1_detailed_deltas.csv",
    "table1_comparison.png",
    "table1_all_models_comparison_grid.png",
    "openrouter_eval_cache.json"
]
for s_file in SYNC_FILES:
    drive_path = f"{DRIVE_OUTPUT_DIR}/{s_file}"
    local_path = f"logs/Table1/{s_file}"
    if os.path.exists(drive_path) and not os.path.exists(local_path):
        shutil.copy(drive_path, local_path)
        print(f"[SYNC DRIVE -> LOCAL] Recuperado com sucesso: {s_file}")


In [ ]:
# Célula 3: Configuração de Credenciais OpenRouter e Verificação dos Datasets
import os
import json

# Tentar obter OpenRouter API Key via Colab Secrets ou arquivo .env
openrouter_key = None
try:
    from google.colab import userdata
    openrouter_key = userdata.get('OPENROUTER_API_KEY')
except Exception:
    pass

if not openrouter_key:
    for env_var in ("OPENROUTER_API_KEY", "OPENAI_API_KEY"):
        val = os.environ.get(env_var)
        if val:
            openrouter_key = val.strip().strip("\"'")
            break

if not openrouter_key and os.path.exists(".env"):
    with open(".env", "r", encoding="utf-8") as f:
        for line in f:
            if "=" in line:
                k, v = line.split("=", 1)
                if k.strip().lower() in ("openrouter_api_key", "openai_api_key"):
                    openrouter_key = v.strip().strip("\"'")
                    break

print("=== Status de Configuração ===")
print(f"OpenRouter API Key configurada: {'✅ Sim' if openrouter_key else '❌ Não (LLMs via OpenRouter serão pulados se vazia)'}")

print("\n--- Verificação dos Datasets de Avaliação ---")
for ds in ["NotInject_one.json", "NotInject_two.json", "NotInject_three.json", "wildguard.json", "BIPIA_text.json", "BIPIA_code.json"]:
    p = os.path.join("datasets", ds)
    print(f"[{'OK' if os.path.exists(p) else 'AUSENTE'}] {p}")


In [ ]:
# Célula 4: Execução da Validação / Trava de Segurança Contra Custos
import os
import shutil

# =============================================================================
#  TRAVA DE SEGURANÇA CONTRA RE-EXECUÇÃO E GASTOS INDEVIDOS DE API
# ==============================================================================
# Se FORCE_RERUN = False:
#   Verifica se os resultados já foram gerados e salvos (no Drive ou em logs/Table1/).
#   Se encontrados, PULA totalmente a inferência, garantindo zero chamadas de API (GPT-4o)
#   e permitindo clicar em 'Executar tudo' no Colab com segurança absoluta.
# Se FORCE_RERUN = True:
#   Força uma nova inferência de todos os modelos e sobrescreve os resultados.
FORCE_RERUN = False

raw_result_path = "logs/Table1/table1_validation_raw.json"
drive_raw_result = f"{DRIVE_OUTPUT_DIR}/table1_validation_raw.json" if 'DRIVE_OUTPUT_DIR' in globals() and os.path.exists(DRIVE_OUTPUT_DIR) else None

# Recupera do Google Drive se ainda não estiver no disco local da VM
if drive_raw_result and os.path.exists(drive_raw_result) and not os.path.exists(raw_result_path):
    os.makedirs("logs/Table1", exist_ok=True)
    shutil.copy(drive_raw_result, raw_result_path)
    print(f"✅ Resultados prévios recuperados do Google Drive: {drive_raw_result}")

if os.path.exists(raw_result_path) and not FORCE_RERUN:
    print("\n" + "="*80)
    print("[TRAVA DE SEGURANÇA ATIVA - INFERÊNCIA PULADA COM SUCESSO]")
    print(f"Resultados de inferência prévios encontrados em: {raw_result_path}")
    print("Zero chamadas de API realizadas. Nenhum crédito de API (OpenRouter/GPT-4o) foi gasto.")
    print("As células seguintes (5, 6, 7 e 8) irão gerar todas as tabelas e gráficos comparativos")
    print("diretamente a partir dos dados salvos.")
    print("(Dica: Se quiser reexecutar uma nova inferência completa, altere 'FORCE_RERUN = True').")
    print("="*80 + "\n")
else:
    print("\n[INICIANDO INFERÊNCIA DOS MODELOS]...")
    key_arg = f"--openrouter_key {openrouter_key}" if openrouter_key else "--skip_openrouter"
    rerun_arg = "--force_rerun" if FORCE_RERUN else ""
    !python3 validate_models_table1.py {key_arg} {rerun_arg} --output_dir logs/Table1


In [ ]:
# Célula 5: Visualização dos Resultados e Tabela Resumo
import os
import json
import shutil
import pandas as pd
from IPython.display import display, HTML, Image

raw_path = "logs/Table1/table1_validation_raw.json"
csv_path = "logs/Table1/table1_validation_summary.csv"

# Se o CSV não existir mas o JSON bruto estiver presente, reconstrói o resumo na hora
if not os.path.exists(csv_path) and os.path.exists(raw_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        raw_d = json.load(f)
    rows = []
    for m_name, m in raw_d.items():
        rows.append({
            "Category": m.get("category", ""),
            "Model": m_name,
            "Over-defense (%)": round(m.get("overdefense", 0.0), 2),
            "Benign (%)": round(m.get("benign", 0.0), 2),
            "Malicious (%)": round(m.get("malicious", 0.0), 2),
            "Average (%)": round(m.get("average", 0.0), 2),
            "GFLOPs": m.get("gflops", ""),
            "Inference (ms)": round(m.get("inference_ms", 0.0), 2),
            "Efficiency": round(m.get("efficiency", 0.0), 2),
        })
    df_gen = pd.DataFrame(rows)
    df_gen.to_csv(csv_path, index=False)

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(HTML("<h3>📊 Tabela 1: Avaliação Comparativa de Performance e Overhead</h3>"))
    display(df)
    
    # Salvar cópia no Google Drive se disponível
    if os.path.exists(DRIVE_OUTPUT_DIR):
        shutil.copy(csv_path, f"{DRIVE_OUTPUT_DIR}/table1_validation_summary.csv")
        if os.path.exists("logs/Table1/table1_validation_raw.json"):
            shutil.copy("logs/Table1/table1_validation_raw.json", f"{DRIVE_OUTPUT_DIR}/table1_validation_raw.json")
        if os.path.exists("logs/Table1/table1_comparison.png"):
            shutil.copy("logs/Table1/table1_comparison.png", f"{DRIVE_OUTPUT_DIR}/table1_comparison.png")
        print(f"\nArquivos de resultado sincronizados com o Google Drive em: {DRIVE_OUTPUT_DIR}")

chart_path = "logs/Table1/table1_comparison.png"
if os.path.exists(chart_path):
    display(HTML("<h3>📈 Visão Geral: Comparação Global de Modelos</h3>"))
    display(Image(chart_path))


## Comparação Direta por Modelo: Artigo (Original) vs Reproduzido

As células a seguir realizam uma comparação minuciosa entre cada resultado experimental reproduzido e os dados oficiais reportados na **Tabela 1 do artigo PIGuard**:
1. **Tabela Detalhada com Deltas ($\Delta$)**: Exibe os valores absolutos do artigo, os valores obtidos na reprodução e a diferença ($\Delta = \text{Reprod} - \text{Artigo}$) para cada uma das 4 métricas (`Over-defense`, `Benign`, `Malicious`, `Average`).
2. **Gráficos Individuais por Modelo**: Gera um gráfico dedicado para **cada modelo**, colocando lado a lado as colunas do **Artigo (Original)** e do **Reproduzido (Obtido)**.
3. **Mosaico Panorâmico (Grid Dashboard)**: Constrói uma visão geral em grade de subplots com todos os modelos lado a lado.

In [ ]:
# Célula 6: Tabela Detalhada de Comparação com Deltas (Artigo vs Reproduzido)
import os
import json
import pandas as pd
from IPython.display import display, HTML

# 1. Valores Oficiais do Artigo (Tabela 1)
PAPER_REF = {
    "Fmops": {"overdefense": 28.32, "benign": 34.63, "malicious": 93.50, "average": 52.15},
    "Deepset": {"overdefense": 29.50, "benign": 34.06, "malicious": 91.50, "average": 51.68},
    "PromptGuard": {"overdefense": 0.29, "benign": 26.82, "malicious": 97.10, "average": 41.40},
    "ProtectAIv2": {"overdefense": 57.23, "benign": 86.20, "malicious": 48.60, "average": 64.01},
    "GPT-4o": {"overdefense": 86.73, "benign": 90.78, "malicious": 79.10, "average": 85.53},
    "Llama-2-chat": {"overdefense": 76.40, "benign": 61.03, "malicious": 31.09, "average": 56.17},
    "LlamaGuard3": {"overdefense": 99.71, "benign": 95.18, "malicious": 28.28, "average": 74.39},
    "PIGuard": {"overdefense": 87.32, "benign": 85.74, "malicious": 77.39, "average": 83.48},
}

# Mapeamento dos modelos avaliados para a referência correspondente no artigo
MODEL_MAPPING = [
    ("Fmops (DistilBERT)", "Fmops (fmops, 2024)", "Fmops"),
    ("Deepset (DeBERTa-v3)", "Deepset (Deepset, 2024b)", "Deepset"),
    ("PromptGuard (Meta 86M)", "PromptGuard (Meta, 2024)", "PromptGuard"),
    ("ProtectAIv2 (DeBERTa-v3)", "ProtectAIv2 (ProtectAI.com, 2024)", "ProtectAIv2"),
    ("GPT-4o (OpenAI)", "GPT-4o (OpenAI, 2024a)", "GPT-4o"),
    ("Llama-2-chat (Touvron et al.)", "Llama-2-chat (Touvron et al., 2023)", "Llama-2-chat"),
    ("LlamaGuard3 (Dubey et al.)", "LlamaGuard3 (Dubey et al., 2024)", "LlamaGuard3"),
    ("PIGuard (Oficial HF - leolee99)", "PIGuard (Oficial HF - leolee99)", "PIGuard"),
    ("PIGuard (train_augmented_mof_original)", "PIGuard (train_augmented_mof_original)", "PIGuard"),
    ("PIGuard (train_augmented_mof_semi_rp)", "PIGuard (train_augmented_mof_semi_rp)", "PIGuard"),
    ("PIGuard (train_augmented_mof_rp)", "PIGuard (train_augmented_mof_rp)", "PIGuard"),
]

# Carrega os resultados brutos
raw_json_path = "logs/Table1/table1_validation_raw.json"
results_raw = {}
if os.path.exists(raw_json_path):
    with open(raw_json_path, "r", encoding="utf-8") as f:
        results_raw = json.load(f)

table_rows = []
for disp_name, raw_key, paper_key in MODEL_MAPPING:
    p_data = PAPER_REF[paper_key]
    r_data = results_raw.get(raw_key, {})
    
    # Fallback para ablação MOF do PIGuard se necessário
    if not r_data and "train_augmented_mof" in raw_key:
        try:
            with open("logs/MOFs/ablation_results_raw.json", "r") as f_abl:
                abl_data = json.load(f_abl).get("evaluated_experiments", {})
                short_k = raw_key.replace("PIGuard (", "").rstrip(")")
                if short_k in abl_data:
                    c = abl_data[short_k]
                    r_data = {
                        "overdefense": c.get("NotInject_overall_acc", 0.0) * 100.0,
                        "benign": c.get("WildGuard_acc", 0.0) * 100.0,
                        "malicious": c.get("BIPIA_overall_acc", 0.0) * 100.0,
                        "average": c.get("Overall_acc", 0.0) * 100.0,
                    }
        except Exception:
            pass

    o_p = p_data["overdefense"]
    b_p = p_data["benign"]
    m_p = p_data["malicious"]
    a_p = p_data["average"]

    o_r = r_data.get("overdefense", None)
    b_r = r_data.get("benign", None)
    m_r = r_data.get("malicious", None)
    a_r = r_data.get("average", None)

    row = {
        "Modelo": disp_name,
        "Over-def (Artigo)": f"{o_p:.2f}%",
        "Over-def (Reprod)": f"{o_r:.2f}%" if o_r is not None else "N/D",
        "Δ Over-def": f"{(o_r - o_p):+.2f}%" if o_r is not None else "-",
        "Benign (Artigo)": f"{b_p:.2f}%",
        "Benign (Reprod)": f"{b_r:.2f}%" if b_r is not None else "N/D",
        "Δ Benign": f"{(b_r - b_p):+.2f}%" if b_r is not None else "-",
        "Malicious (Artigo)": f"{m_p:.2f}%",
        "Malicious (Reprod)": f"{m_r:.2f}%" if m_r is not None else "N/D",
        "Δ Malicious": f"{(m_r - m_p):+.2f}%" if m_r is not None else "-",
        "Average (Artigo)": f"{a_p:.2f}%",
        "Average (Reprod)": f"{a_r:.2f}%" if a_r is not None else "N/D",
        "Δ Average": f"{(a_r - a_p):+.2f}%" if a_r is not None else "-",
    }
    table_rows.append(row)

df_deltas = pd.DataFrame(table_rows)
display(HTML("<h3>📋 Comparação Lado a Lado: Artigo vs Reproduzido com Deltas (Δ)</h3>"))
display(df_deltas)

# Salvar tabela com deltas
df_deltas.to_csv("logs/Table1/table1_detailed_deltas.csv", index=False)
if os.path.exists(DRIVE_OUTPUT_DIR):
    df_deltas.to_csv(f"{DRIVE_OUTPUT_DIR}/table1_detailed_deltas.csv", index=False)
    print(f"Tabela com deltas sincronizada com o Google Drive em: {DRIVE_OUTPUT_DIR}/table1_detailed_deltas.csv")


In [ ]:
# Célula 7: Gráficos Individuais por Modelo (Colunas Artigo vs Reproduzido)
import os
import shutil
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML

# Salvaguarda de definições se executado fora de ordem
if "PAPER_REF" not in globals() or "MODEL_MAPPING" not in globals():
    PAPER_REF = {
        "Fmops": {"overdefense": 28.32, "benign": 34.63, "malicious": 93.50, "average": 52.15},
        "Deepset": {"overdefense": 29.50, "benign": 34.06, "malicious": 91.50, "average": 51.68},
        "PromptGuard": {"overdefense": 0.29, "benign": 26.82, "malicious": 97.10, "average": 41.40},
        "ProtectAIv2": {"overdefense": 57.23, "benign": 86.20, "malicious": 48.60, "average": 64.01},
        "GPT-4o": {"overdefense": 86.73, "benign": 90.78, "malicious": 79.10, "average": 85.53},
        "Llama-2-chat": {"overdefense": 76.40, "benign": 61.03, "malicious": 31.09, "average": 56.17},
        "LlamaGuard3": {"overdefense": 99.71, "benign": 95.18, "malicious": 28.28, "average": 74.39},
        "PIGuard": {"overdefense": 87.32, "benign": 85.74, "malicious": 77.39, "average": 83.48},
    }
    MODEL_MAPPING = [
        ("Fmops (DistilBERT)", "Fmops (fmops, 2024)", "Fmops"),
        ("Deepset (DeBERTa-v3)", "Deepset (Deepset, 2024b)", "Deepset"),
        ("PromptGuard (Meta 86M)", "PromptGuard (Meta, 2024)", "PromptGuard"),
        ("ProtectAIv2 (DeBERTa-v3)", "ProtectAIv2 (ProtectAI.com, 2024)", "ProtectAIv2"),
        ("GPT-4o (OpenAI)", "GPT-4o (OpenAI, 2024a)", "GPT-4o"),
        ("Llama-2-chat (Touvron et al.)", "Llama-2-chat (Touvron et al., 2023)", "Llama-2-chat"),
        ("LlamaGuard3 (Dubey et al.)", "LlamaGuard3 (Dubey et al., 2024)", "LlamaGuard3"),
        ("PIGuard (Oficial HF - leolee99)", "PIGuard (Oficial HF - leolee99)", "PIGuard"),
        ("PIGuard (train_augmented_mof_original)", "PIGuard (train_augmented_mof_original)", "PIGuard"),
        ("PIGuard (train_augmented_mof_semi_rp)", "PIGuard (train_augmented_mof_semi_rp)", "PIGuard"),
        ("PIGuard (train_augmented_mof_rp)", "PIGuard (train_augmented_mof_rp)", "PIGuard"),
    ]
if "results_raw" not in globals():
    results_raw = {}
    if os.path.exists("logs/Table1/table1_validation_raw.json"):
        with open("logs/Table1/table1_validation_raw.json", "r", encoding="utf-8") as f:
            results_raw = json.load(f)

CHARTS_DIR = "logs/Table1/individual_charts"
os.makedirs(CHARTS_DIR, exist_ok=True)
DRIVE_CHARTS_DIR = f"{DRIVE_OUTPUT_DIR}/individual_charts" if 'DRIVE_OUTPUT_DIR' in globals() and os.path.exists(DRIVE_OUTPUT_DIR) else None
if DRIVE_CHARTS_DIR:
    os.makedirs(DRIVE_CHARTS_DIR, exist_ok=True)

metrics = ["Over-defense", "Benign", "Malicious", "Average"]
width = 0.35

display(HTML("<h3>📊 Gráficos Comparativos Individuais por Modelo</h3><p>Para cada modelo avaliado, as colunas exibem lado a lado os resultados do <b>Artigo (Original)</b> e os obtidos na <b>Reprodução</b>.</p>"))

for disp_name, raw_key, paper_key in MODEL_MAPPING:
    p_data = PAPER_REF[paper_key]
    r_data = results_raw.get(raw_key, {})
    
    # Fallback para ablação se necessário
    if not r_data and "train_augmented_mof" in raw_key:
        try:
            with open("logs/MOFs/ablation_results_raw.json", "r") as f_abl:
                abl_data = json.load(f_abl).get("evaluated_experiments", {})
                short_k = raw_key.replace("PIGuard (", "").rstrip(")")
                if short_k in abl_data:
                    c = abl_data[short_k]
                    r_data = {
                        "overdefense": c.get("NotInject_overall_acc", 0.0) * 100.0,
                        "benign": c.get("WildGuard_acc", 0.0) * 100.0,
                        "malicious": c.get("BIPIA_overall_acc", 0.0) * 100.0,
                        "average": c.get("Overall_acc", 0.0) * 100.0,
                    }
        except Exception:
            pass

    paper_vals = [
        p_data["overdefense"],
        p_data["benign"],
        p_data["malicious"],
        p_data["average"],
    ]

    reprod_vals = [
        r_data.get("overdefense", 0.0),
        r_data.get("benign", 0.0),
        r_data.get("malicious", 0.0),
        r_data.get("average", 0.0),
    ]

    has_reprod = any(v > 0 for v in reprod_vals)
    if not has_reprod:
        continue

    fig, ax = plt.subplots(figsize=(8.5, 5), dpi=150)
    x = np.arange(len(metrics))

    rects1 = ax.bar(x - width/2, paper_vals, width, label="Artigo (Original)", color="#1D3557", edgecolor="black", linewidth=0.6)
    rects2 = ax.bar(x + width/2, reprod_vals, width, label="Reproduzido (Obtido)", color="#E63946", edgecolor="black", linewidth=0.6)

    # Diferença no Average
    avg_delta = reprod_vals[3] - paper_vals[3]
    delta_symbol = "▲" if avg_delta >= 0 else "▼"

    ax.set_ylabel("Acurácia (%)", fontsize=11, fontweight="bold")
    ax.set_title(
        f"{disp_name}\nArtigo (Original) vs Reproduzido (Obtido) | Δ Médio: {avg_delta:+.2f}% {delta_symbol}",
        fontsize=13,
        fontweight="bold",
        pad=14
    )
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 118)
    ax.legend(frameon=True, facecolor="#F8F9FA", edgecolor="#ADB5BD", fontsize=10, loc="upper right")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    # Rótulos nas colunas do artigo
    for rect in rects1:
        h = rect.get_height()
        ax.annotate(f"{h:.1f}%", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 4),
                    textcoords="offset points", ha="center", va="bottom", fontsize=9.5, fontweight="bold", color="#1D3557")

    # Rótulos nas colunas reproduzidas com diferença
    for i, rect in enumerate(rects2):
        h = rect.get_height()
        diff = h - paper_vals[i]
        diff_str = f"({diff:+.1f}%)" if abs(diff) >= 0.05 else ""
        ax.annotate(f"{h:.1f}%\n{diff_str}", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 4),
                    textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold", color="#E63946")

    plt.tight_layout()

    # Salva arquivo individual
    slug = disp_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_").replace(".", "_")
    chart_file = os.path.join(CHARTS_DIR, f"compare_{slug}.png")
    plt.savefig(chart_file, dpi=200, bbox_inches="tight")
    if DRIVE_CHARTS_DIR:
        shutil.copy(chart_file, os.path.join(DRIVE_CHARTS_DIR, f"compare_{slug}.png"))

    plt.show()
    plt.close()

print(f"\nTodos os gráficos individuais foram salvos em: {CHARTS_DIR}")
if DRIVE_CHARTS_DIR:
    print(f"Gráficos individuais sincronizados com o Google Drive em: {DRIVE_CHARTS_DIR}")


In [ ]:
# Célula 8: Mosaico Panorâmico (Grid Comparativo de Todos os Modelos)
import os
import json
import math
import shutil
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML

# Salvaguarda de definições se executado fora de ordem
if "PAPER_REF" not in globals() or "MODEL_MAPPING" not in globals():
    PAPER_REF = {
        "Fmops": {"overdefense": 28.32, "benign": 34.63, "malicious": 93.50, "average": 52.15},
        "Deepset": {"overdefense": 29.50, "benign": 34.06, "malicious": 91.50, "average": 51.68},
        "PromptGuard": {"overdefense": 0.29, "benign": 26.82, "malicious": 97.10, "average": 41.40},
        "ProtectAIv2": {"overdefense": 57.23, "benign": 86.20, "malicious": 48.60, "average": 64.01},
        "GPT-4o": {"overdefense": 86.73, "benign": 90.78, "malicious": 79.10, "average": 85.53},
        "Llama-2-chat": {"overdefense": 76.40, "benign": 61.03, "malicious": 31.09, "average": 56.17},
        "LlamaGuard3": {"overdefense": 99.71, "benign": 95.18, "malicious": 28.28, "average": 74.39},
        "PIGuard": {"overdefense": 87.32, "benign": 85.74, "malicious": 77.39, "average": 83.48},
    }
    MODEL_MAPPING = [
        ("Fmops (DistilBERT)", "Fmops (fmops, 2024)", "Fmops"),
        ("Deepset (DeBERTa-v3)", "Deepset (Deepset, 2024b)", "Deepset"),
        ("PromptGuard (Meta 86M)", "PromptGuard (Meta, 2024)", "PromptGuard"),
        ("ProtectAIv2 (DeBERTa-v3)", "ProtectAIv2 (ProtectAI.com, 2024)", "ProtectAIv2"),
        ("GPT-4o (OpenAI)", "GPT-4o (OpenAI, 2024a)", "GPT-4o"),
        ("Llama-2-chat (Touvron et al.)", "Llama-2-chat (Touvron et al., 2023)", "Llama-2-chat"),
        ("LlamaGuard3 (Dubey et al.)", "LlamaGuard3 (Dubey et al., 2024)", "LlamaGuard3"),
        ("PIGuard (Oficial HF - leolee99)", "PIGuard (Oficial HF - leolee99)", "PIGuard"),
        ("PIGuard (train_augmented_mof_original)", "PIGuard (train_augmented_mof_original)", "PIGuard"),
        ("PIGuard (train_augmented_mof_semi_rp)", "PIGuard (train_augmented_mof_semi_rp)", "PIGuard"),
        ("PIGuard (train_augmented_mof_rp)", "PIGuard (train_augmented_mof_rp)", "PIGuard"),
    ]
if "results_raw" not in globals():
    results_raw = {}
    if os.path.exists("logs/Table1/table1_validation_raw.json"):
        with open("logs/Table1/table1_validation_raw.json", "r", encoding="utf-8") as f:
            results_raw = json.load(f)

# Filtra apenas modelos com dados válidos
valid_models = []
for disp_name, raw_key, paper_key in MODEL_MAPPING:
    p_data = PAPER_REF[paper_key]
    r_data = results_raw.get(raw_key, {})
    if not r_data and "train_augmented_mof" in raw_key:
        try:
            with open("logs/MOFs/ablation_results_raw.json", "r") as f_abl:
                abl_data = json.load(f_abl).get("evaluated_experiments", {})
                short_k = raw_key.replace("PIGuard (", "").rstrip(")")
                if short_k in abl_data:
                    c = abl_data[short_k]
                    r_data = {
                        "overdefense": c.get("NotInject_overall_acc", 0.0) * 100.0,
                        "benign": c.get("WildGuard_acc", 0.0) * 100.0,
                        "malicious": c.get("BIPIA_overall_acc", 0.0) * 100.0,
                        "average": c.get("Overall_acc", 0.0) * 100.0,
                    }
        except Exception:
            pass

    reprod_vals = [
        r_data.get("overdefense", 0.0),
        r_data.get("benign", 0.0),
        r_data.get("malicious", 0.0),
        r_data.get("average", 0.0),
    ]
    if any(v > 0 for v in reprod_vals):
        paper_vals = [
            p_data["overdefense"],
            p_data["benign"],
            p_data["malicious"],
            p_data["average"],
        ]
        valid_models.append((disp_name, paper_vals, reprod_vals))

n_models = len(valid_models)
ncols = 2
nrows = math.ceil(n_models / ncols)

fig_grid, axes_grid = plt.subplots(nrows, ncols, figsize=(16, 4.2 * nrows), dpi=200)
axes_grid = axes_grid.flatten()

metrics_abbr = ["Overdef", "Benign", "Malicious", "Average"]
width = 0.35

for idx, (disp_name, p_vals, r_vals) in enumerate(valid_models):
    ax = axes_grid[idx]
    x_g = np.arange(len(metrics_abbr))

    ax.bar(x_g - width/2, p_vals, width, label="Artigo", color="#1D3557", edgecolor="black", linewidth=0.5)
    ax.bar(x_g + width/2, r_vals, width, label="Reproduzido", color="#E63946", edgecolor="black", linewidth=0.5)

    delta_avg = r_vals[3] - p_vals[3]
    ax.set_title(f"{disp_name} (Δ Avg: {delta_avg:+.1f}%)", fontsize=11, fontweight="bold")
    ax.set_xticks(x_g)
    ax.set_xticklabels(metrics_abbr, fontsize=10)
    ax.set_ylim(0, 115)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    if idx == 0:
        ax.legend(loc="upper right", fontsize=9)

    for i_g in range(len(metrics_abbr)):
        ax.annotate(f"{p_vals[i_g]:.0f}%", xy=(x_g[i_g] - width/2, p_vals[i_g]), xytext=(0, 2),
                    textcoords="offset points", ha="center", va="bottom", fontsize=8, color="#1D3557", fontweight="bold")
        ax.annotate(f"{r_vals[i_g]:.0f}%", xy=(x_g[i_g] + width/2, r_vals[i_g]), xytext=(0, 2),
                    textcoords="offset points", ha="center", va="bottom", fontsize=8, color="#E63946", fontweight="bold")

# Ocultar eixos sobressalentes se houver
for idx in range(n_models, len(axes_grid)):
    fig_grid.delaxes(axes_grid[idx])

plt.suptitle("Dashboard Consolidado: Artigo (Original) vs Reproduzido em Todos os Modelos", fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()

grid_chart_path = "logs/Table1/table1_all_models_comparison_grid.png"
plt.savefig(grid_chart_path, dpi=250, bbox_inches="tight")
if 'DRIVE_OUTPUT_DIR' in globals() and os.path.exists(DRIVE_OUTPUT_DIR):
    shutil.copy(grid_chart_path, f"{DRIVE_OUTPUT_DIR}/table1_all_models_comparison_grid.png")
    print(f"\nMosaico consolidado sincronizado com o Google Drive em: {DRIVE_OUTPUT_DIR}/table1_all_models_comparison_grid.png")

plt.show()
